# End-to-End Sound Generation on Real Audio Data (ALD-SC)

This notebook trains the full ALD-SC pipeline end-to-end on the audio files in `data/` (NSynth-style .wav files). It uses the real frozen `EnCodec` encoder and the graph-structured decoder.

The dataset is split into **train / validation / test** sets so that evaluation is honest and reproducible.

## Pipeline
1. Load .wav files from `data/` and split into train/val/test.
2. Extract real EnCodec latents from the **training set** and build the frozen ArrowSpace prior.
3. Train graph decoder vs. matched-capacity baseline decoder on the training set.
4. Train a 1-D DiT denoiser on the training set.
5. Evaluate on the validation and test sets.
6. Sample, decode, and ablate.

The dataset is split into **train / validation / test** sets so that evaluation is honest and reproducible.

> LSD is a **sound-generation** variant, not the ALD-SC research artifact. There is
> no falsifiable claim here: the goal is sound production for its own sake. Training
> is intentionally **non-repeatable** when `SEED = None`, and latent-space noise
> injection (`NOISE_INJECT`) is provided as a creative axis — each run yields a
> different model and a different sound. Use this to explore *the sound of the future*.

In [1]:
# --- Generation knobs ---
# Set SEED = None for a non-repeatable run: each execution samples a
# fresh random seed, so every model (and every generated sound) differs.
# This is intentional: non-repeatable training is a *feature* for artistic
# exploration, not a bug.
SEED = 3407

# --- Artistic noise injection ---
# Add Gaussian noise to the EnCodec latent z before decoding during
# decoder training. 0.0 reproduces the deterministic baseline; larger
# values raise the early loss and produce different, more varied models
# across runs. Tune for taste.
NOISE_INJECT = 0.1

STEPS = 50
TEMPERATURE = 0.85
USE_C_SPEC = True

# --- Dataset knobs ---
from pathlib import Path

# Resolve data dir relative to the repo root so the notebook runs from
# either the repo root or the notebooks/ subdirectory.
DATA_DIR = Path.cwd().parent / 'data'   # directory containing .wav files (NSynth-style)
AUDIO_LENGTH = 96000        # 4 seconds @ 24kHz (NSynth clips are 4s)
SAMPLE_RATE = 24000
SUBSET_SIZE = 256           # set to None to use all files

# --- Split knobs ---
TRAIN_FRAC = 0.7            # fraction of data for training
VAL_FRAC = 0.15             # fraction of data for validation
# remainder goes to test

# --- Model knobs ---
Q = 8                       # prior chart dimension
K = 4                       # prior knn
BASE_CHANNELS = 32

# --- Training knobs ---
DECODER_EPOCHS = 20
DIFFUSION_EPOCHS = 20
BATCH_SIZE = 8
LR = 1e-3

LATENT_LENGTH = AUDIO_LENGTH // 320  # EnCodec 24kHz stride
print(f'Data dir: {DATA_DIR}')
print(f'Audio: {AUDIO_LENGTH/SAMPLE_RATE:.1f}s ({AUDIO_LENGTH} samples) -> latent length {LATENT_LENGTH}')
print(f'Subset size: {SUBSET_SIZE}')
print(f'Noise inject: {NOISE_INJECT}')
print(f'Split: train={TRAIN_FRAC}, val={VAL_FRAC}, test={1-TRAIN_FRAC-VAL_FRAC:.2f}')
print(f'Knobs: seed={SEED}, steps={STEPS}, temp={TEMPERATURE}, use_c_spec={USE_C_SPEC}')

Data dir: /Users/tuned-silicon/papers/latent-sound-diffusion/data
Audio: 4.0s (96000 samples) -> latent length 300
Subset size: 256
Split: train=0.7, val=0.15, test=0.15
Knobs: seed=3407, steps=50, temp=0.85, use_c_spec=True


## Imports

In [2]:
import random
from pathlib import Path

import torch
import torch.nn as nn
import torchaudio
from IPython.display import Audio, display

from ald_sc.build_prior import build_arrow_prior
from ald_sc.audio_codec import EnCodecEncoder, AudioVAE, BaselineAudioDecoder
from ald_sc.graph_decoder import GraphDecoder
from ald_sc.dit import MinimalDiT
from ald_sc.data import AudioFolderDataset, build_audio_dataloader
from ald_sc.losses import ALDSCLoss
from ald_sc.schedule import CosineSchedule
from ald_sc.sampling import sample_ddim
from ald_sc.trainer import train_audio_decoder, train_audio_diffusion, log_training

device = torch.device('cpu')
if SEED is None:
    import time as _time
    SEED = int(_time.time()) % (2**31)
    print(f'Non-repeatable run: sampled SEED={SEED}')
random.seed(SEED)
torch.manual_seed(SEED)
print('Imports done. Device:', device)
print('EnCodecEncoder loaded lazily — first encode call may download weights.')

Imports done. Device: cpu
EnCodecEncoder loaded lazily — first encode call may download weights.


## Step 1: Load Real Audio Dataset and Split Train/Val/Test

In [3]:
# List all .wav files in the data directory
all_files = sorted(Path(DATA_DIR).glob('*.wav'))
print(f'Found {len(all_files)} audio files in {DATA_DIR}')

# Optional reproducible subset for faster demo runs
if SUBSET_SIZE is not None and len(all_files) > SUBSET_SIZE:
    random.seed(SEED)
    selected_files = random.sample(all_files, SUBSET_SIZE)
else:
    selected_files = all_files

# Reproducible train/val/test split
random.seed(SEED)
random.shuffle(selected_files)
n = len(selected_files)
n_train = int(n * TRAIN_FRAC)
n_val = int(n * VAL_FRAC)
train_files = selected_files[:n_train]
val_files = selected_files[n_train:n_train + n_val]
test_files = selected_files[n_train + n_val:]

print(f'Total selected: {n}')
print(f'  Train: {len(train_files)}')
print(f'  Val:   {len(val_files)}')
print(f'  Test:  {len(test_files)}')

# AudioFolderDataset loads, resamples to 24kHz, and crops/pads to AUDIO_LENGTH.
# A single dataset is constructed and its file list is overridden per split
# (three independent instances avoid any cross-split state sharing).
assert len(train_files) > 0, "Train split is empty; decrease TRAIN_FRAC or SUBSET_SIZE"
assert len(val_files) > 0, "Val split is empty; decrease VAL_FRAC or SUBSET_SIZE"
assert len(test_files) > 0, "Test split is empty; adjust TRAIN_FRAC/VAL_FRAC or SUBSET_SIZE"

train_dataset = AudioFolderDataset(root=str(DATA_DIR), audio_length=AUDIO_LENGTH, sample_rate=SAMPLE_RATE)
train_dataset.files = train_files
val_dataset = AudioFolderDataset(root=str(DATA_DIR), audio_length=AUDIO_LENGTH, sample_rate=SAMPLE_RATE)
val_dataset.files = val_files
test_dataset = AudioFolderDataset(root=str(DATA_DIR), audio_length=AUDIO_LENGTH, sample_rate=SAMPLE_RATE)
test_dataset.files = test_files

print(f'Train clip shape: {train_dataset[0].shape}')

Found 9811 audio files in /Users/tuned-silicon/papers/latent-sound-diffusion/data
Total selected: 256
  Train: 179
  Val:   38
  Test:  39
Train clip shape: torch.Size([1, 96000])


## Step 2: Build the ArrowSpace Prior from Training-Set EnCodec Features

In [4]:
# Real frozen EnCodec encoder
encoder = EnCodecEncoder(sample_rate=SAMPLE_RATE, bandwidth=24)

# Build the prior from the training set only
train_loader = build_audio_dataloader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)

features = []
for batch in train_loader:
    z = encoder.extract_features(batch)
    features.append(z.mean(dim=2))
embeddings = torch.cat(features, dim=0)
print(f'Train corpus EnCodec embeddings: {embeddings.shape}')

# Build the frozen ArrowSpace prior
prior = build_arrow_prior(embeddings, q=Q, k=K)
print(f'L_F: {prior.L_F.shape}, U_q: {prior.U_q.shape}, q={prior.q}')

/Users/tuned-silicon/papers/latent-sound-diffusion/.venv/lib/python3.13/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Train corpus EnCodec embeddings: torch.Size([176, 128])
L_F: torch.Size([128, 128]), U_q: torch.Size([128, 8]), q=8


## Step 3: Train Graph Decoder vs. Baseline Decoder

In [5]:
# Graph decoder (uses real EnCodec latents + ArrowSpace prior)
graph_decoder = GraphDecoder(
    latent_channels=128,
    out_channels=1,
    feature_dim=128,
    base_channels=BASE_CHANNELS,
    prior=prior,
    upsample_strides=(2, 4, 5, 8),
)

# Matched-capacity baseline decoder (no graph structure)
baseline_decoder = BaselineAudioDecoder(
    latent_channels=128,
    out_channels=1,
    base_channels=BASE_CHANNELS,
    upsample_strides=(2, 4, 5, 8),
)

loss_fn = ALDSCLoss(
    prior=prior,
    lambda_rec=1.0,
    lambda_stft=0.0,
    lambda_chart=0.5,
    lambda_smooth=0.1,
)

# Train loaders
train_loader = build_audio_dataloader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = build_audio_dataloader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Train graph decoder
graph_vae = AudioVAE(encoder=encoder, decoder=graph_decoder)
print('Training graph decoder on real EnCodec latents...')
graph_losses = list(log_training(
    train_audio_decoder(
        train_loader, graph_vae, prior, loss_fn,
        epochs=DECODER_EPOCHS, lr=LR, device=device,
        noise_std=NOISE_INJECT,
    ),
    label='Graph decoder',
))
print(f'  Overall: {graph_losses[0]["loss"]:.4f} -> {graph_losses[-1]["loss"]:.4f}')

# Train baseline decoder
baseline_vae = AudioVAE(encoder=encoder, decoder=baseline_decoder)
print('Training baseline decoder...')
baseline_losses = list(log_training(
    train_audio_decoder(
        train_loader, baseline_vae, prior, loss_fn,
        epochs=DECODER_EPOCHS, lr=LR, device=device,
        noise_std=NOISE_INJECT,
    ),
    label='Baseline decoder',
))
print(f'  Overall: {baseline_losses[0]["loss"]:.4f} -> {baseline_losses[-1]["loss"]:.4f}')

Training graph decoder on real EnCodec latents...
2026-08-02T12:00:40.027632Z [info     ] epoch                          epoch=0 label='Graph decoder' mean_loss=0.1850520826198838 steps=22
2026-08-02T12:00:50.373696Z [info     ] epoch                          epoch=1 label='Graph decoder' mean_loss=0.1631653146310286 steps=22
2026-08-02T12:01:00.739970Z [info     ] epoch                          epoch=2 label='Graph decoder' mean_loss=0.15723301030018114 steps=22
2026-08-02T12:01:11.012080Z [info     ] epoch                          epoch=3 label='Graph decoder' mean_loss=0.15844189951365645 steps=22
2026-08-02T12:01:21.157202Z [info     ] epoch                          epoch=4 label='Graph decoder' mean_loss=0.1563280922445384 steps=22
2026-08-02T12:01:31.428915Z [info     ] epoch                          epoch=5 label='Graph decoder' mean_loss=0.15855000371282751 steps=22
2026-08-02T12:01:41.808642Z [info     ] epoch                          epoch=6 label='Graph decoder' mean_loss=0.

## Step 4: Train the 1-D DiT Denoiser

In [6]:
dit = MinimalDiT(
    latent_channels=128,
    latent_length=LATENT_LENGTH,
    patch_size=8,
    dim=64,
    depth=2,
    num_heads=4,
    spec_dim=3 * Q,
)
sched = CosineSchedule(num_steps=1000)

# Freeze VAE for diffusion training
for p in graph_vae.parameters():
    p.requires_grad_(False)

print(f'Training 1-D DiT on real EnCodec latents (latent_length={LATENT_LENGTH})...')
diff_losses = list(log_training(
    train_audio_diffusion(
        train_loader, graph_vae, dit, prior, sched,
        epochs=DIFFUSION_EPOCHS, lr=LR, device=device,
    ),
    label='DiT',
))
print(f'  Overall: {diff_losses[0]["loss"]:.4f} -> {diff_losses[-1]["loss"]:.4f}')

Training 1-D DiT on real EnCodec latents (latent_length=300)...
2026-08-02T12:07:28.032765Z [info     ] epoch                          epoch=0 label=DiT mean_loss=12.63480333848433 steps=22
2026-08-02T12:07:35.924056Z [info     ] epoch                          epoch=1 label=DiT mean_loss=8.943167187950827 steps=22
2026-08-02T12:07:43.817197Z [info     ] epoch                          epoch=2 label=DiT mean_loss=6.158201466907155 steps=22
2026-08-02T12:07:51.827106Z [info     ] epoch                          epoch=3 label=DiT mean_loss=4.339933958920565 steps=22
2026-08-02T12:07:59.950599Z [info     ] epoch                          epoch=4 label=DiT mean_loss=3.2901124954223633 steps=22
2026-08-02T12:08:07.921986Z [info     ] epoch                          epoch=5 label=DiT mean_loss=3.0310982140627774 steps=22
2026-08-02T12:08:15.921895Z [info     ] epoch                          epoch=6 label=DiT mean_loss=2.805132275277918 steps=22
2026-08-02T12:08:23.878219Z [info     ] epoch       

## Step 5: Generate Sound

In [7]:
# Sample latent z from noise
torch.manual_seed(SEED)
dit = dit.eval()
z = sample_ddim(dit, sched, batch_size=1, steps=STEPS, seed=SEED, device=device)

# Apply temperature scaling
z = z * TEMPERATURE
print(f'Sampled z: {z.shape}')

# Derive c_spec from z (self-consistent decoding)
a = z.mean(dim=2)
c_spec = prior.chart_energy_descriptor(a)

# Decode with graph decoder
with torch.no_grad():
    if USE_C_SPEC:
        audio_graph = graph_decoder(z, c_spec)
    else:
        audio_graph = graph_decoder(z, torch.zeros_like(c_spec))
    audio_baseline = baseline_decoder(z)

# Normalize for playback
def normalize(audio):
    audio = audio.squeeze(0)
    peak = audio.abs().max()
    if peak > 0:
        audio = audio / peak
    return audio

audio_graph_norm = normalize(audio_graph)
audio_baseline_norm = normalize(audio_baseline)

print(f'Graph decoder audio: {audio_graph_norm.shape}, {audio_graph_norm.shape[-1]/SAMPLE_RATE:.2f}s')
print(f'Baseline decoder audio: {audio_baseline_norm.shape}, {audio_baseline_norm.shape[-1]/SAMPLE_RATE:.2f}s')

Sampled z: torch.Size([1, 128, 300])
Graph decoder audio: torch.Size([1, 96000]), 4.00s
Baseline decoder audio: torch.Size([1, 96000]), 4.00s


In [8]:
print('Graph decoder output:')
display(Audio(audio_graph_norm.numpy(), rate=SAMPLE_RATE))

Graph decoder output:


In [9]:
print('Baseline decoder output:')
display(Audio(audio_baseline_norm.numpy(), rate=SAMPLE_RATE))

Baseline decoder output:


In [10]:
print('Real training clip (reference):')
real_clip = train_dataset[0]  # (1, T)
display(Audio(real_clip.numpy(), rate=SAMPLE_RATE))

Real training clip (reference):


## Step 6: Evaluate on Validation and Test Sets

In [11]:
def eval_reconstruction(vae, loader, loss_fn, device):
    vae.eval()
    total_rec, total_chart, n = 0, 0, 0
    with torch.no_grad():
        for batch in loader:
            x = batch.to(device)
            z, A, c_spec, x_hat = vae(x, prior)
            losses = loss_fn(x, x_hat, A, A.detach())
            total_rec += losses['rec'].item()
            total_chart += losses['chart'].item()
            n += 1
    return {'rec': total_rec/n, 'chart': total_chart/n}

# Validation-set comparison
print('=== Validation-set Reconstruction Comparison ===')
graph_val = eval_reconstruction(graph_vae, val_loader, loss_fn, device)
baseline_val = eval_reconstruction(baseline_vae, val_loader, loss_fn, device)
print(f'Graph decoder:    L1={graph_val["rec"]:.6f}  chart={graph_val["chart"]:.6f}')
print(f'Baseline decoder: L1={baseline_val["rec"]:.6f}  chart={baseline_val["chart"]:.6f}')
diff = baseline_val['rec'] - graph_val['rec']
print(f'Graph improvement: {diff:+.6f} (positive = graph is better)')

# Test-set final evaluation
test_loader = build_audio_dataloader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print('\n=== Test-set Final Evaluation ===')
graph_test = eval_reconstruction(graph_vae, test_loader, loss_fn, device)
baseline_test = eval_reconstruction(baseline_vae, test_loader, loss_fn, device)
print(f'Graph decoder:    L1={graph_test["rec"]:.6f}  chart={graph_test["chart"]:.6f}')
print(f'Baseline decoder: L1={baseline_test["rec"]:.6f}  chart={baseline_test["chart"]:.6f}')
diff = baseline_test['rec'] - graph_test['rec']
print(f'Graph improvement: {diff:+.6f} (positive = graph is better)')

=== Validation-set Reconstruction Comparison ===
Graph decoder:    L1=0.141883  chart=0.000000
Baseline decoder: L1=0.137057  chart=0.000000
Graph improvement: -0.004826 (positive = graph is better)

=== Test-set Final Evaluation ===
Graph decoder:    L1=0.134168  chart=0.000000
Baseline decoder: L1=0.127469  chart=0.000000
Graph improvement: -0.006699 (positive = graph is better)


## Step 7: lambda_ED Ablation (Validation Set)

In [12]:
class NoCSPecVAE(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, x, prior):
        z, a, c_spec = self.encoder.encode(x, prior)
        zero_cspec = torch.zeros_like(c_spec)
        x_hat = self.decoder(z, zero_cspec)
        return z, a, c_spec, x_hat

ablation_vae = NoCSPecVAE(encoder, graph_decoder)
print('=== lambda_ED Ablation (validation set) ===')
ablation_val = eval_reconstruction(ablation_vae, val_loader, loss_fn, device)
print(f'With c_spec:    L1={graph_val["rec"]:.6f}  chart={graph_val["chart"]:.6f}')
print(f'Without c_spec: L1={ablation_val["rec"]:.6f}  chart={ablation_val["chart"]:.6f}')
diff = ablation_val['rec'] - graph_val['rec']
print(f'lambda_ED effect: {diff:+.6f} (positive = gating helps)')

=== lambda_ED Ablation (validation set) ===
With c_spec:    L1=0.141883  chart=0.000000
Without c_spec: L1=0.141865  chart=0.000000
lambda_ED effect: -0.000018 (positive = gating helps)


## Summary

In [13]:
print('=== Summary ===')
print(f'Encoder: real EnCodec 24kHz (frozen)')
print(f'Dataset: {DATA_DIR} (train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)})')
print(f'Prior: ArrowSpace q={Q}, k={K} (built from training set only)')
print(f'Graph decoder loss: {graph_losses[-1]["loss"]:.4f}')
print(f'Baseline decoder loss: {baseline_losses[-1]["loss"]:.4f}')
print(f'DiT loss: {diff_losses[-1]["loss"]:.4f}')
print('\nRe-run the knob cell with different SUBSET_SIZE, TRAIN_FRAC, VAL_FRAC,')
print('SEED, STEPS, TEMPERATURE, or USE_C_SPEC to explore the generated sound space.')

=== Summary ===
Encoder: real EnCodec 24kHz (frozen)
Dataset: /Users/tuned-silicon/papers/latent-sound-diffusion/data (train=179, val=38, test=39)
Prior: ArrowSpace q=8, k=4 (built from training set only)
Graph decoder loss: 0.0831
Baseline decoder loss: 0.1354
DiT loss: 2.8400

Re-run the knob cell with different SUBSET_SIZE, TRAIN_FRAC, VAL_FRAC,
SEED, STEPS, TEMPERATURE, or USE_C_SPEC to explore the generated sound space.
